# Building a Basic Chatbot with LangGraph

In this notebook, we will build a basic chatbot (local LLM calls only) using `LangGraph`, so you can get familiar with the APIs and its structure.

## Install dependencies

In [ ]:
!sudo apt update
!sudo apt install -y python3-dev graphviz libgraphviz-dev pkg-config

In [ ]:
!pip install langgraph langchain langchain-ollama pygraphviz langchain-openai

## Launch vLLM in the Background

Execute the cell below to create the vllm_serve.sh

In [ ]:
vllm_file = f"""#!/bin/bash

VLLM_USE_TRITON_FLASH_ATTN=0 \\
vllm serve Qwen/Qwen3-30B-A3B \\
    --served-model-name Qwen3-30B-A3B \\
    --api-key abc-123 \\
    --port 8000 \\
    --enable-auto-tool-choice \\
    --tool-call-parser hermes \\
    --trust-remote-code 2>&1 | tee vllm_serve.log
"""

with open('vllm_server.sh', 'w', encoding='utf-8') as f:
    f.write(vllm_file)

Open a new terminal to execute the `vllm_serve.sh` file. This will serve an LLM locally.

In Jupyter, open a new terminal. `File > New > Terminal`, copy the content below and execute it.

```sh
bash vllm_serve.sh
```

Now, the LLM will be ready to be used once you see `Application startup complete.`

## Create a `StateGraph`

We start by creating our `State` class that it is used to build the `StateGraph` object. The `StateGraph` defines the structure of our chatbot, this is the nodes and edges. We are adding `nodes` to represent the LLM and functions/tools our chatbot can call and `edges` to specify the transitions between nodes.

In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    """
    Messages have the type 'list'.
    The `add_messages` function defines how this state key should be updated
    (in this case, it appends messages to the list, rather than overwriting them)
    """
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

The graph handles two important tasks
- Each node will receive the current `State` as input and update the state as output.
- Updates to the `messages` attribute will be appended to the existing list. This is called [reducer functions](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers)

## Add a Node

Now, we can add a node to the graph. `Nodes` represent units of work.

In [ ]:
from langchain.chat_models import init_chat_model

In [ ]:
llm = init_chat_model(
    model='Qwen3-30B-A3B',
    model_provider='openai',
    base_url='http://localhost:8000/v1',
    api_key='abc-123'
)

Let's add the llm call into a simple node. 

Note how the `chatbot` function takes the `State` as input and returns a dictionary containing and updated `message` list.

The `.add_node` takes at least two arguments
1. Unique node name
2. object that will be called when the node is used.

In [ ]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}


graph_builder.add_node("chatbot", chatbot)

## Add an `entry` point

LangGraph works with directed graphs, so we need to specify the starting point for our graph each time it is run. Note that we use the keyword `START` and the node name `chatbot`.

In [ ]:
graph_builder.add_edge(START, "chatbot")


## Add an `exit` point

Similarly, we need to indicate the node where the graph finishes execution. Note that we use the keyword `END` and the node name `chatbot`.

In [ ]:
graph_builder.add_edge("chatbot", END)

## Compile the Graph

Once all the nodes and edges are added, we need to compile our graph.

In [ ]:
graph = graph_builder.compile()

## Visualize the graph

We can visualize the graph to make sure the nodes and edges are correct. 

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_png()))

## Run the Chatbot

With the graph compiled, we can now run it. Let's create a function that takes an user string and returns the response from the model.

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

In [ ]:
stream_graph_updates('What are Large Language Models?')

----------
Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT